> **⚠️ Windows에서 TensorFlow 실행 시 "DLL load failed" 오류가 나면:**  
> 1) [Google Colab](https://colab.research.google.com)에서 노트북을 열어 실행하거나,  
> 2) [Microsoft Visual C++ Redistributable](https://aka.ms/vs/17/release/vc_redist.x64.exe) (최신) 설치 후 재시도해 보세요.

# [LAB12] 딥러닝 > 신경망의 이해 > 03. 학습효율 개선 - 퍼셉트론(XOR)

두 개 이상의 신경망 층을 쌓는 경우에 대한 예제

텐서플로우의 학습성능을 개선하기 위해 **콜백함수**를 사용할 수 있음

**콜백함수**: 모델의 학습 방향, 저장 시점, 학습 정지 시점 등에 관한 상황을 설정 하기 위한 도구

## 📘 #01. 준비작업

### 📝 [1] 패키지 참조

In [ ]:
from hossam import *
import os
from pandas import DataFrame, concat
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, RMSprop
from tensorflow.keras.losses import mse
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tqdm.keras import TqdmCallback

### 📝 [2] 데이터셋 준비

#### ✏ 데이터 가져오기

#### ✏ origin을 10배로 증강 (데이터 수가 매우 적으므로 학습이 가능한 수준으로 증강)

In [ ]:
origin = load_data("logical_xor")
origin

In [ ]:
df = concat([origin] * 10, ignore_index=True)
df.describe()

## 📘 #02. 탐색적 데이터 분석

## 📘 #03. 데이터 전처리

훈련/검증 데이터 분할

## 📘 #04 신경망 모델 적합

### 📝 [1] 학습 모델 구성

| 구분 | 모델 | 활성화 함수 | 옵티마이저 | 손실함수 | 평가지표 | 과적합 판정 지표 | 대표예제 |
|------|------|-------------|------------|----------|----------|------------------|----------|
| 퍼셉트론 XOR Gate | relu→sigmoid | Adam | binary_crossentropy | accuracy | Val Loss − Train Loss | XOR Gate |

In [ ]:
yname = "target"
x = df.drop(columns=[yname])
y = df[yname]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=52)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
rows, cols = x_train.shape
print(rows, cols)

In [ ]:
model = Sequential()
model.add(Input(shape=(cols,)))
model.add(Dense(4, activation="relu"))
model.add(Dense(1, activation="sigmoid"))
model.compile(
    optimizer="SGD",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

### 📝 [2] 학습하기

#### ✏ 콜백함수 사용 방법

#### ✏ 콜백함수 종류: EarlyStopping(), ReduceLROnPlateau(), ModelCheckpoint()

#### ✏ CheckPoint가 저장될 경로

In [ ]:
cwd = os.getcwd()
target_dir = os.path.join(cwd, "tensorflow_checkpoint")
if not os.path.exists(target_dir):
    os.mkdir(target_dir)
checkpoint_path = os.path.join(target_dir, "model04-cp-{epoch:04d}-ckpt.keras")
checkpoint_path

## 📘 #05 성능평가

### 📝 [1] 성능평가 지표

In [ ]:
%%time
result = model.fit(
    x_train, y_train,
    epochs=500,
    validation_data=(x_test, y_test),
    verbose=0,
    callbacks=[
        TqdmCallback(verbose=1),
        ModelCheckpoint(filepath=checkpoint_path, monitor='val_loss', verbose=0, save_best_only=True),
        EarlyStopping(monitor='val_loss', patience=5, min_delta=0.0001),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0, verbose=1)
    ]
)
result

In [ ]:
# Train 성능 평가
train_eval = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
test_eval = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
final_results = DataFrame([train_eval, test_eval])
final_results.insert(0, "Dataset", ["Train", "Test"])
final_results["loss_gap"] = None
final_results.loc[1, "loss_gap"] = final_results.loc[1, "loss"] - final_results.loc[0, "loss"]
final_results

### 📝 [2] 학습 과정 확인

### 📝 [3] Loss(MSE) 학습곡선

In [ ]:
history_df = DataFrame(data=result.history)
history_df["epoch"] = history_df.index + 1
history_df.head()

In [ ]:
figsize = (1280 / 100, 720 / 100)
fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=100)
sb.lineplot(data=history_df, x="epoch", y="loss", ax=ax, label="Train Loss")
sb.lineplot(data=history_df, x="epoch", y="val_loss", ax=ax, label="Validation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training vs Validation Loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

## 📘 #06. 학습 결과 적용

In [ ]:
r = model.predict(x_test, verbose=0)
r

In [ ]:
y_pred = r.reshape(-1) > 0.5
y_df = DataFrame({"y_true": y_test, "y_pred": y_pred})
y_df['y_pred'] = y_df['y_pred'].astype(int)
y_df